# v4 vs V1 — Side by Side on Published Benchmarks (GPU)

**V1** is the previous team's architecture (`tabjoint_vishal_averaged_experiments`, vendored unchanged in
[`v1/`](./v1)): 8 MLPs on random feature subsets → bidirectional Transformer with a `[CLS]` token, trained with a
CKA diversity penalty. **v4** is the current architecture, with the per-feature embeddings added after the
Grinsztajn diagnostics.

Both are trained on the **same rows, the same splits and the same metric**, next to a tuned XGBoost, a plain MLP,
and the **published results** for 33 models on exactly these splits.

| Dataset | Rows | Task | Metric |
|---|---|---|---|
| Higgs Small (HI) | 98,049 | binary | accuracy ↑ |
| Facebook Comments (FB) | 197,080 | regression | RMSE ↓ (original units) |
| Santander (SA) | 200,000 | binary | accuracy ↑ (90% one class → 0.899 is the floor) |

Source of splits and published numbers: Gorishniy, Rubachev & Babenko, *On Embeddings for Numerical Features in
Tabular Deep Learning*, NeurIPS 2022 (arXiv:2203.05556).

### Fairness notes, stated up front
* **V1 runs with its own defaults** (`hparams.py`), and v4 with the settings from notebook 06. Neither is tuned per
  dataset, while every published model was. Both of ours are handicapped the same way.
* **V1's own training loop** is used for V1 (its optimiser, early stopping, CKA penalty), so this measures V1 as
  written, not a reimplementation.
* Metrics are recomputed on the full test split for every model, because V1's internal scoring averages per-batch
  metrics (for RMSE that isn't even the right number).

## 0. Setup — device first

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd(); sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch, optuna
import matplotlib.pyplot as plt, seaborn as sns
from joblib import Parallel, delayed

from benchmarks import BENCHMARKS, load_benchmark, published, higher_is_better
from training import get_device
from v1_runner import V1_DEFAULTS

DEVICE = get_device()                      # CUDA when available; SRP_DEVICE=cpu overrides
# Knobs, overridable from the shell so a headless run needs no edits:
#   SRP_SUBSAMPLE=2000 SRP_SEEDS=1 SRP_XGB_TRIALS=2 jupyter nbconvert --execute ...
KEYS   = os.environ.get("SRP_KEYS", "HI,FB,SA").split(",")
SEEDS  = tuple(range(int(os.environ.get("SRP_SEEDS", "3"))))
XGB_TRIALS = int(os.environ.get("SRP_XGB_TRIALS", "25"))
SUBSAMPLE  = int(os.environ.get("SRP_SUBSAMPLE", "0")) or None
EPOCHS     = int(os.environ.get("SRP_EPOCHS", "100"))
V4_TRAIN_KW = dict(epochs=EPOCHS, lr=1e-3, batch_size=512, weight_decay=1e-4, patience=10)
EMB_KW = dict(d_embedding=8, n_frequencies=16, sigma=0.1)
ENS_KW = dict(num_learners=16, hidden_dim=16, embed_dim=32, depth=1, dropout=0.1, feature_frac=0.7)
ATT_KW = dict(embed_dim=32, num_heads=4, attn_depth=2, ff_mult=4, dropout=0.1, need_weights=False)

# Models to run. (label, kind, aggregator, embedding mode)
MODELS = [
    ("V1 (predecessors)",      "v1",    None,            None),
    ("v4 mean-pool",           "v4",    "meanpool_wide", "none"),
    ("v4 mean-pool + PLR",     "v4",    "meanpool_wide", "periodic"),
    ("v4 causal attn",         "v4",    "causal",        "none"),
    ("v4 causal attn + PLR",   "v4",    "causal",        "periodic"),
    ("PlainMLP",               "mlp",   None,            None),
]
# Several runs share one GPU comfortably (each well under 1 GB); on CPU run smaller jobs side by side.
THREADS_PER_JOB = 2
# On one GPU, 3 jobs fit comfortably now that validation is evaluated in chunks. Lower it with
# SRP_JOBS=1 if the card is shared with someone else's large job.
N_JOBS = int(os.environ.get("SRP_JOBS", "0")) or (
    3 if DEVICE.type == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB)))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
# One log per run, kept: an interrupted run's partial results are evidence too.
# runs/07_progress.log always points at the newest.
STAMP = time.strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(RUN_DIR, f"07_progress_{STAMP}.log")
_latest = os.path.join(RUN_DIR, "07_progress.log")
if os.path.islink(_latest) or os.path.exists(_latest):
    os.remove(_latest)
os.symlink(os.path.basename(LOG_PATH), _latest)
def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")
def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

sns.set_theme(style="whitegrid", context="notebook")
optuna.logging.set_verbosity(optuna.logging.WARNING)
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"
log(f"device: {DEVICE} ({gpu}) | torch {torch.__version__} | {N_JOBS} parallel jobs")
if DEVICE.type != "cuda":
    print("NOTE: running on CPU. V1 and the larger v4 arms take hours here — this notebook is meant for the GPU server.")

## 1. Load the three benchmarks (the paper's exact splits)

In [ ]:
DATA = {}
for k in KEYS:
    d = load_benchmark(k)
    if SUBSAMPLE:
        n = SUBSAMPLE
        d.X_tr, d.y_tr = d.X_tr[:n], d.y_tr[:n]
        d.X_va, d.y_va = d.X_va[:n // 2], d.y_va[:n // 2]
        d.X_te, d.y_te = d.X_te[:n], d.y_te[:n]
    DATA[k] = d
    log(d.summary())
METRIC = {k: ("rmse" if DATA[k].task == "regression" else "acc") for k in KEYS}
if SUBSAMPLE:
    print(f"SMOKE TEST on {SUBSAMPLE} rows — numbers below are not comparable to the paper.")

## 2. One scoring function for everybody

Accuracy for the binary datasets, RMSE **in the target's original units** for Facebook Comments (the paper trains on
a standardised target and reports RMSE in original units, so we map back the same way).

In [ ]:
def score_from_raw(data, y_true, raw):
    # raw = logits for binary classification, standardised predictions for regression
    raw = np.asarray(raw).reshape(len(y_true), -1)
    if data.task == "regression":
        return float(np.sqrt(np.mean((raw[:, 0] - y_true) ** 2)) * data.y_std)
    if data.task == "binary":
        return float(((raw[:, 0] > 0).astype(int) == y_true).mean())
    return float((raw.argmax(1) == y_true).mean())

print("published single-model results on these splits (for reference)")
FOCUS = ["CatBoost", "XGBoost", "MLP", "MLP-PLR", "ResNet", "Transformer-PLR"]
show(pd.DataFrame({BENCHMARKS[k]["label"]: {m: f"{published(k, m)[0]:.3f} ± {published(k, m)[1]:.3f}"
                                            for m in FOCUS} for k in KEYS}))

## 3. Train everything — V1 and v4 side by side

In [ ]:
def run_one(label, kind, agg, mode, key, seed, arrays, task_kind, output_dim, n_features,
            ens_kw, att_kw, emb_kw, train_kw, v1_hp, device_str, project_dir, threads):
    import sys, time
    if project_dir not in sys.path: sys.path.insert(0, project_dir)
    import numpy as np, torch
    torch.set_num_threads(threads)
    from training import train_model, predict, to_tensors
    t0 = time.time()

    if kind == "v1":
        from v1_runner import run_v1
        class _Task:      # v1_runner only needs arrays(), task, output_dim, n_features
            pass
        t = _Task(); t.arrays = lambda: arrays; t.task = task_kind
        t.output_dim = output_dim; t.n_features = n_features
        r = run_v1(t, seed=seed, device=device_str, hp=v1_hp, threads=threads)
        return {"label": label, "key": key, "seed": seed, "raw": r["test_pred"],
                "n_params": r["n_params"], "seconds": r["seconds"], "epochs": r["epochs_run"]}

    from weak_learners import WeakLearnerConfig
    from attention import AttentionConfig
    from embeddings import EmbeddingConfig
    from model import make_arm, PlainMLP
    data = to_tensors(arrays, device_str)
    torch.manual_seed(seed); np.random.seed(seed)
    if kind == "mlp":
        model = PlainMLP(n_features, task=task_kind, output_dim=output_dim)
    else:
        model = make_arm(agg, n_features, WeakLearnerConfig(**ens_kw, seed=seed, batched=True),
                         AttentionConfig(**att_kw), task=task_kind, output_dim=output_dim,
                         embedding=EmbeddingConfig(mode=mode, **emb_kw))
    model, hist = train_model(model, data, seed=seed, **train_kw)
    return {"label": label, "key": key, "seed": seed, "raw": predict(model, data["Xte"]),
            "n_params": sum(p.numel() for p in model.parameters()),
            "seconds": time.time() - t0, "epochs": hist["best_epoch"] + 1}

jobs = [(lbl, kind, agg, mode, k, s) for k in KEYS for (lbl, kind, agg, mode) in MODELS for s in SEEDS]
jobs.sort(key=lambda j: j[1] != "v1")        # V1 first: it is the slowest
log(f"{len(jobs)} runs on {N_JOBS} workers ({DEVICE}) ...")

t0, RES = time.time(), []
for (lbl, kind, agg, mode, k, s), r in zip(jobs, Parallel(n_jobs=N_JOBS, return_as="generator")(
        delayed(run_one)(lbl, kind, agg, mode, k, s, DATA[k].arrays(), DATA[k].task, DATA[k].output_dim,
                         DATA[k].n_features, ENS_KW, ATT_KW, EMB_KW, V4_TRAIN_KW, {}, str(DEVICE),
                         PROJECT_DIR, THREADS_PER_JOB)
        for lbl, kind, agg, mode, k, s in jobs)):
    d = DATA[k]
    r["score"] = score_from_raw(d, d.y_te, r.pop("raw"))
    RES.append(r)
    log(f"  [{len(RES):3d}/{len(jobs)}] {BENCHMARKS[k]['label']:18s} {lbl:22s} seed {s}  "
        f"{METRIC[k]} {r['score']:.4f}  ({r['seconds']/60:.1f} min)  elapsed {(time.time()-t0)/60:.1f} min")
R = pd.DataFrame(RES)
log(f"all runs done in {(time.time()-t0)/60:.1f} min")

## 4. XGBoost on the same splits

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
XGB = {}
for k in KEYS:
    d, a, t0 = DATA[k], DATA[k].arrays(), time.time()
    hib = higher_is_better(k)
    def make(params, seed):
        common = dict(tree_method="hist", device=str(DEVICE), random_state=seed,
                      n_estimators=2000, early_stopping_rounds=50, **params)
        return XGBRegressor(**common) if d.task == "regression" else XGBClassifier(**common)
    def raw(m, X):
        if d.task == "regression": return m.predict(X)
        p = np.clip(m.predict_proba(X)[:, 1], 1e-7, 1 - 1e-7); return np.log(p / (1 - p))
    def objective(trial):
        params = {"max_depth": trial.suggest_int("max_depth", 3, 10),
                  "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),
                  "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                  "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                  "min_child_weight": trial.suggest_float("min_child_weight", 1e-4, 100.0, log=True),
                  "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True)}
        m = make(params, 0).fit(a["Xtr"], a["ytr"], eval_set=[(a["Xva"], a["yva"])], verbose=False)
        return score_from_raw(d, d.y_va, raw(m, a["Xva"]))
    st = optuna.create_study(direction="maximize" if hib else "minimize",
                             sampler=optuna.samplers.TPESampler(seed=0, multivariate=True))
    st.optimize(objective, n_trials=XGB_TRIALS)
    XGB[k] = np.array([score_from_raw(d, d.y_te, raw(make(st.best_params, s).fit(
        a["Xtr"], a["ytr"], eval_set=[(a["Xva"], a["yva"])], verbose=False), a["Xte"])) for s in SEEDS])
    log(f"[xgb] {BENCHMARKS[k]['label']:18s} ours {XGB[k].mean():.4f} | published {published(k, 'XGBoost')[0]:.3f} "
        f"({(time.time()-t0)/60:.1f} min)")

## 5. Results

In [ ]:
# groupby+unstack rather than pivot_table: with a single seed the std is all-NaN and
# pivot_table silently DROPS the column, which then breaks every lookup below.
g = R.groupby(["label", "key"])["score"]
piv = g.mean().unstack("key")
sdv = g.std(ddof=1).unstack("key").reindex_like(piv).fillna(0.0)
order = [m[0] for m in MODELS]
rows = []
for lbl in order:
    row = {"model": lbl, "params (HI)": R[(R.label == lbl) & (R.key == "HI")]["n_params"].iloc[0]}
    for k in KEYS:
        row[BENCHMARKS[k]["label"]] = f"{piv.loc[lbl, k]:.4f} ± {sdv.loc[lbl, k]:.4f}"
    rows.append(row)
rows.append({"model": "XGBoost (ours, tuned)", "params (HI)": np.nan,
             **{BENCHMARKS[k]["label"]: f"{XGB[k].mean():.4f} ± {XGB[k].std(ddof=1):.4f}" for k in KEYS}})
for m in ("XGBoost", "CatBoost", "MLP", "MLP-PLR"):
    rows.append({"model": f"{m} (published)", "params (HI)": np.nan,
                 **{BENCHMARKS[k]["label"]: f"{published(k, m)[0]:.3f} ± {published(k, m)[1]:.3f}" for k in KEYS}})
print("Test scores — accuracy for HI/SA (higher better), RMSE for FB (lower better)")
show(pd.DataFrame(rows).set_index("model"), {"params (HI)": "{:,.0f}"})

In [ ]:
fig, axes = plt.subplots(1, len(KEYS), figsize=(6 * len(KEYS), 5.6))
for ax, k in zip(np.atleast_1d(axes), KEYS):
    hib = higher_is_better(k)
    entries = [(lbl, piv.loc[lbl, k], sdv.loc[lbl, k],
                "#7b5cff" if lbl.startswith("v4") else "#2b8a3e" if lbl.startswith("V1") else "#4c8dff")
               for lbl in order]
    entries.append(("XGBoost (ours)", XGB[k].mean(), XGB[k].std(ddof=1), "#ff8a3d"))
    for m in ("XGBoost", "CatBoost", "MLP-PLR"):
        mu, sd = published(k, m)
        entries.append((f"{m} (published)", mu, sd, "#9aa0a6"))
    entries.sort(key=lambda e: e[1], reverse=not hib)
    for i, (lbl, mu, sd, c) in enumerate(entries):
        ax.errorbar(mu, i, xerr=(0 if np.isnan(sd) else sd), fmt="o", color=c, ms=8, capsize=4,
                    mec="black" if lbl.startswith(("v4", "V1")) else c)
        ax.text(mu, i + 0.3, f"{mu:.3f}", ha="center", fontsize=8, color=c)
    ax.set_yticks(range(len(entries))); ax.set_yticklabels([e[0] for e in entries], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(f"test {METRIC[k]} ({'higher' if hib else 'lower'} better)")
    ax.set_title(BENCHMARKS[k]["label"], fontweight="bold")
plt.suptitle("purple = v4 · green = V1 (predecessors) · orange = our XGBoost · grey = published",
             y=1.02, fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Findings

In [ ]:
sep = "=" * 96
print(sep); print("FINDINGS — v4 vs V1 on published benchmarks"); print(sep)
best_v4 = {k: min(((piv.loc[l, k], l) for l in order if l.startswith("v4")),
                  key=lambda z: z[0] if not higher_is_better(k) else -z[0]) for k in KEYS}

print("\n1. v4 vs V1 (the head-to-head)")
for k in KEYS:
    hib, v1 = higher_is_better(k), piv.loc["V1 (predecessors)", k]
    v4, lbl = best_v4[k]
    better = (v4 > v1) if hib else (v4 < v1)
    print(f"   {BENCHMARKS[k]['label']:18s} V1 {v1:.4f} | best v4 {v4:.4f} ({lbl}) -> "
          f"{'v4 better' if better else 'V1 better'} by {abs(v4 - v1):.4f}")

print("\n2. Did per-feature embeddings help here too?")
for k in KEYS:
    for agg in ("mean-pool", "causal attn"):
        a, b = piv.loc[f"v4 {agg}", k], piv.loc[f"v4 {agg} + PLR", k]
        d = (b - a) if higher_is_better(k) else (a - b)
        print(f"   {BENCHMARKS[k]['label']:18s} {agg:12s} {a:.4f} -> {b:.4f}  ({d:+.4f} in favour of PLR)")

print("\n3. Against XGBoost and the published models")
for k in KEYS:
    hib = higher_is_better(k); v4, lbl = best_v4[k]
    sign = 1 if hib else -1
    print(f"   {BENCHMARKS[k]['label']:18s} best v4 {v4:.4f} | our XGBoost {XGB[k].mean():.4f} "
          f"({sign*(v4-XGB[k].mean()):+.4f}) | published XGBoost {published(k,'XGBoost')[0]:.3f} "
          f"| published best NN (MLP-PLR) {published(k,'MLP-PLR')[0]:.3f}")

print("\n4. Sanity: does our XGBoost reproduce the published one?")
for k in KEYS:
    pub = published(k, "XGBoost")
    d = XGB[k].mean() - pub[0]
    print(f"   {BENCHMARKS[k]['label']:18s} ours {XGB[k].mean():.4f} vs published {pub[0]:.3f} ({d:+.4f})")
print("\n" + sep)

## 7. Notes

* Neither v4 nor V1 was tuned per dataset here; the published models all were. Tuning ours (the Optuna machinery in
  `tuning.py` works on any of these arms) is the obvious follow-up if the gap is close.
* 3 seeds here vs 15 in the paper, so our standard deviations are rough.
* If our XGBoost lands near the published XGBoost, the data pipeline and metrics line up with the paper and the
  comparison is apples-to-apples.